In [ ]:
# https://grok.com/chat/d1306a5a-1c52-4ed6-83e3-52821e16c4a7


import numpy as np
import json
from pathlib import Path
import logging
from scipy.signal import butter, filtfilt
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import tensorflow as tf
from sklearn.metrics import mean_squared_error, confusion_matrix, classification_report
import seaborn as sns
from typing import List, Tuple, Dict, Optional

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Constants
NOISE_LEVELS = [
    ("clean", None),
    ("mild", dict(gaussian_std_ratio=0.03, baseline_amp=0.05, powerline_amp=0.05, motion_amp=0.3)),
    ("moderate", dict(gaussian_std_ratio=0.08, baseline_amp=0.12, powerline_amp=0.10, motion_amp=0.5)),
    ("severe", dict(gaussian_std_ratio=0.15, baseline_amp=0.25, powerline_amp=0.15, motion_amp=1.2))
]
BASELINE_FREQ = 0.33
POWERLINE_FREQ = 50
MOTION_PROB = 0.008
MOTION_DURATION_MIN, MOTION_DURATION_MAX = 10, 100
FS = 200
SEGMENT_LENGTH = 1024
OVERLAP = 0.0
DUOMENU_APLANKAS = Path.home() / 'DI/ZIVEO_2025/DUOMENYS_UPD'
PARAMETRU_APLANKAS = DUOMENU_APLANKAS / 'PARAMETRAI'
MODEL_PATH = PARAMETRU_APLANKAS / 'resunet_ecg_1024_05_31.keras'
CLEAN_ECG_DIR = DUOMENU_APLANKAS / 'clean_ecg_for_test'
PORTION_LENGTH_IN_SECS = 10  # seconds
MSE_THRESH = 0.01  # Example threshold, adjust as needed
FLAG_PLOT = True  # Set to False to disable plotting

def bandpass_filter(signal: np.ndarray, low: float = 0.5, high: float = 40, fs: int = 200, order: int = 2) -> np.ndarray:
    b, a = butter(order, [low, high], btype='bandpass', fs=fs)
    return filtfilt(b, a, signal)

def normalize(signal: np.ndarray) -> np.ndarray:
    mean = np.mean(signal)
    std = np.std(signal)
    if std < 1e-8:
        logger.warning("Signal has near-zero variance; returning zeros")
        return np.zeros_like(signal)
    return (signal - mean) / std

def add_gaussian_noise(signal: np.ndarray, std_ratio: float = 0.05) -> np.ndarray:
    return signal + np.random.normal(0, std_ratio * np.std(signal), size=signal.shape)

def add_baseline_wander(signal: np.ndarray, freq: float = 0.33, amp: float = 0.1, fs: int = 200) -> np.ndarray:
    t = np.arange(len(signal)) / fs
    return signal + amp * np.sin(2 * np.pi * freq * t)

def add_powerline(signal: np.ndarray, freq: float = 50, amp: float = 0.05, fs: int = 200) -> np.ndarray:
    t = np.arange(len(signal)) / fs
    return signal + amp * np.sin(2 * np.pi * freq * t)

def add_motion_artifact(signal: np.ndarray, prob: float = MOTION_PROB, amp: float = 0.5, 
                       duration_min: int = 10, duration_max: int = 50) -> np.ndarray:
    artifact = signal.copy()
    i = 0
    while i < len(artifact):
        if np.random.rand() < prob:
            duration = min(np.random.randint(duration_min, duration_max+1), len(artifact) - i)
            artifact[i:i+duration] += amp * (np.random.rand() - 0.5)
            i += duration
        else:
            i += 1
    return artifact

def apply_noise(signal: np.ndarray, noise_params: Dict) -> np.ndarray:
    x = add_gaussian_noise(signal, noise_params['gaussian_std_ratio'])
    x = add_baseline_wander(x, BASELINE_FREQ, noise_params['baseline_amp'], fs=FS)
    x = add_powerline(x, POWERLINE_FREQ, noise_params['powerline_amp'], fs=FS)
    x = add_motion_artifact(
        x, prob=MOTION_PROB, amp=noise_params['motion_amp'],
        duration_min=MOTION_DURATION_MIN, duration_max=MOTION_DURATION_MAX
    )
    return x

def calculate_snr(clean: np.ndarray, noisy: np.ndarray) -> float:
    signal_power = np.mean(clean ** 2)
    noise_power = np.mean((noisy - clean) ** 2)
    return 10 * np.log10(signal_power / noise_power) if noise_power > 0 else float('inf')

def calculate_correlation(clean: np.ndarray, noisy: np.ndarray) -> float:
    return np.corrcoef(clean, noisy)[0, 1]

def get_ecg_noise_indices_annotated(json_path: Path) -> List[Tuple[int, int]]:
    """Load annotated noise indices from JSON file."""
    try:
        with open(json_path, 'r', encoding='UTF-8') as f:
            data = json.load(f)
        noise_indices_from_json = data.get('noises_annotated', [])
        noise_indices = [(item['startIndex'], item['endIndex']) for item in noise_indices_from_json]
        logger.debug(f"Loaded annotated noise indices from {json_path}: {noise_indices}")
        return noise_indices
    except (json.JSONDecodeError, FileNotFoundError, KeyError) as e:
        logger.warning(f"Failed to load noise indices from {json_path}: {e}. Assuming no noise regions.")
        return []

def load_and_preprocess_ecg(ecg_path: Path) -> Tuple[np.ndarray, List[Tuple[int, int]]]:
    try:
        signal = np.load(ecg_path)
    except Exception as e:
        logger.error(f"Failed to load ECG file {ecg_path}: {e}")
        return np.array([]), []
    
    signal = bandpass_filter(signal, fs=FS)
    noise_indices = get_ecg_noise_indices_annotated(ecg_path.with_suffix('.json'))
    return signal, noise_indices

def segment_signal(signal: np.ndarray, segment_length: int, overlap: float, 
                  noise_indices: Optional[List[Tuple[int, int]]] = None) -> List[Tuple[int, int]]:
    """Segment signal into overlapping windows, excluding noisy regions and invalid segments."""
    if noise_indices is None:
        noise_indices = []
    
    if len(signal) < segment_length:
        logger.warning(f"Signal too short ({len(signal)} < {segment_length}); skipping")
        return []
    
    step = int(segment_length * (1 - overlap))
    segments = []
    
    for start in range(0, len(signal) - segment_length + 1, step):
        end = start + segment_length
        
        # Check if segment overlaps with any noise region
        overlaps_noise = False
        for noise_start, noise_end in noise_indices:
            if not (end <= noise_start or start >= noise_end):
                overlaps_noise = True
                logger.debug(f"Skipping segment [{start}:{end}] due to overlap with noise [{noise_start}:{noise_end}]")
                break
        if overlaps_noise:
            continue
        segments.append((start, end))
    
    return segments

def evaluate_noise(model, mse_thresh: float, ecg_path: Path, noise_option: str = 'random', 
                  noise_probs: Optional[List[float]] = None) -> Tuple[Dict, np.ndarray]:
    # Load and preprocess ECG
    signal, noise_indices = load_and_preprocess_ecg(ecg_path)
    
    if len(signal) == 0:
        logger.error(f"No valid signal loaded from {ecg_path}")
        return {
            'segments': [],
            'mse_values': {level: [] for level, _ in NOISE_LEVELS},
            'snr_values': {level: [] for level, _ in NOISE_LEVELS},
            'corr_values': {level: [] for level, _ in NOISE_LEVELS},
            'counts': {level: 0 for level, _ in NOISE_LEVELS}
        }, np.array([])
    
    # Segment signal
    segments = segment_signal(signal, SEGMENT_LENGTH, OVERLAP, noise_indices)
    
    if not segments:
        logger.warning(f"No valid segments found for {ecg_path}")
        return {
            'segments': [],
            'mse_values': {level: [] for level, _ in NOISE_LEVELS},
            'snr_values': {level: [] for level, _ in NOISE_LEVELS},
            'corr_values': {level: [] for level, _ in NOISE_LEVELS},
            'counts': {level: 0 for level, _ in NOISE_LEVELS}
        }, np.zeros_like(signal)
    
    # Results storage
    results = {
        'segments': [],
        'mse_values': {level: [] for level, _ in NOISE_LEVELS},
        'snr_values': {level: [] for level, _ in NOISE_LEVELS},
        'corr_values': {level: [] for level, _ in NOISE_LEVELS},
        'counts': {level: 0 for level, _ in NOISE_LEVELS}
    }
    
    # Process segments
    reconstructed_signal = np.zeros_like(signal)
    segment_counts = np.zeros(len(signal))
    
    for start, end in segments:
        # Extract and normalize segment
        segment = signal[start:end]
        clean_segment = normalize(segment)
        
        # Assign noise level
        if noise_option == 'random':
            if noise_probs is None:
                noise_probs = [0.25] * 4  # Equal probability
            noise_idx = np.random.choice(len(NOISE_LEVELS), p=noise_probs)
        elif noise_option == 'random_select':
            noise_idx = np.random.randint(len(NOISE_LEVELS))
        else:  # Specific level
            noise_idx = [i for i, (level, _) in enumerate(NOISE_LEVELS) if level == noise_option][0]
            
        noise_level, noise_params = NOISE_LEVELS[noise_idx]
        results['counts'][noise_level] += 1
        
        # Apply noise if not clean
        noisy_segment = clean_segment if noise_params is None else apply_noise(clean_segment, noise_params)
        noisy_segment = normalize(noisy_segment)
        
        # Autoencoder prediction
        input_segment = noisy_segment.reshape(1, SEGMENT_LENGTH, 1)
        predicted = model.predict(input_segment, verbose=0).reshape(-1)
        mse = mean_squared_error(noisy_segment, predicted)
        
        # Store metrics
        results['mse_values'][noise_level].append(mse)
        results['snr_values'][noise_level].append(calculate_snr(clean_segment, noisy_segment))
        results['corr_values'][noise_level].append(calculate_correlation(clean_segment, noisy_segment))
        
        # Detection result
        is_noisy = mse > mse_thresh
        results['segments'].append({
            'start': start,
            'end': end,
            'noise_level': noise_level,
            'noise_idx': noise_idx,
            'mse': mse,
            'is_noisy': is_noisy
        })
        
        # Reconstruct signal
        reconstructed_signal[start:end] += noisy_segment
        segment_counts[start:end] += 1
    
    # Average reconstructed signal
    mask = segment_counts > 0
    reconstructed_signal[mask] /= segment_counts[mask]
    
    return results, reconstructed_signal

def visualize_results(signal: np.ndarray, results: Dict, output_path: Path):
    if len(signal) == 0 or not results['segments']:
        logger.warning("No signal or segments to visualize")
        return
    
    # Calculate portion length as a multiple of SEGMENT_LENGTH
    base_portion_length = int(PORTION_LENGTH_IN_SECS * FS)  # Base length in samples
    num_segments = round(base_portion_length / SEGMENT_LENGTH)  # Nearest integer number of segments
    portion_length = num_segments * SEGMENT_LENGTH  # Adjust to multiple of SEGMENT_LENGTH
    
    colors = {'clean': 'green', 'mild': 'yellow', 'moderate': 'orange', 'severe': 'red'}
    
    # Calculate number of portions
    signal_length = len(signal)
    num_portions = (signal_length + portion_length - 1) // portion_length
    
    for portion_idx in range(num_portions):
        start = portion_idx * portion_length
        end = min(start + portion_length, signal_length)
        t = np.arange(start, end) / FS
        
        plt.figure(figsize=(15, 6))
        # Plot signal
        signal_line, = plt.plot(t, signal[start:end], 'b-', alpha=0.3)
        
        # Plot segments within this portion
        for seg in results['segments']:
            seg_start, seg_end = seg['start'], seg['end']
            # Check if segment overlaps with current portion
            if seg_end > start and seg_start < end:
                # Adjust segment boundaries to fit within portion
                plot_start = max(seg_start, start)
                plot_end = min(seg_end, end)
                t_seg = np.arange(plot_start, plot_end) / FS
                plt.fill_between(t_seg, signal[plot_start:plot_end], 
                               color=colors[seg['noise_level']], alpha=0.3)
                
                if seg['is_noisy']:
                    plt.text(t_seg.mean(), signal[plot_start:plot_end].max(), 
                           'N', color='black', ha='center')
        
        # Create fixed legend with explicit handles and labels
        handles = [
            mpatches.Patch(color='blue', alpha=0.3, label='Signal'),
            mpatches.Patch(color='green', alpha=0.3, label='clean noise'),
            mpatches.Patch(color='yellow', alpha=0.3, label='mild noise'),
            mpatches.Patch(color='orange', alpha=0.3, label='moderate noise'),
            mpatches.Patch(color='red', alpha=0.3, label='severe noise')
        ]
        
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.title(f'ECG Signal Portion {portion_idx + 1} ({start/FS:.1f}s - {end/FS:.1f}s) start: {start}, end: {end}')
        plt.legend(handles=handles)
        plt.savefig(output_path / f'signal_portion_{portion_idx + 1}.png')
        plt.close()
    
    # Plot MSE distribution
    plt.figure(figsize=(10, 6))
    for level, mse_values in results['mse_values'].items():
        if mse_values:
            sns.kdeplot(mse_values, label=level)
    plt.xlabel('MSE')
    plt.ylabel('Density')
    plt.title('MSE Distribution by Noise Level')
    plt.legend()
    plt.savefig(output_path / 'mse_distribution.png')
    plt.close()

def print_constants():
    """Print script constants in a human-readable format."""
    print("Script Constants:")
    print("=" * 40)
    
    print("NOISE_LEVELS:")
    for level, params in NOISE_LEVELS:
        if params is None:
            print(f"  - {level}: No parameters")
        else:
            print(f"  - {level}:")
            for param, value in params.items():
                print(f"      {param}: {value}")
    
    print(f"BASELINE_FREQ: {BASELINE_FREQ} Hz")
    print(f"POWERLINE_FREQ: {POWERLINE_FREQ} Hz")
    print(f"MOTION_PROB: {MOTION_PROB}")
    print(f"MOTION_DURATION_MIN: {MOTION_DURATION_MIN} samples")
    print(f"MOTION_DURATION_MAX: {MOTION_DURATION_MAX} samples")
    print(f"FS: {FS} Hz (sampling frequency)")
    print(f"SEGMENT_LENGTH: {SEGMENT_LENGTH} samples")
    print(f"OVERLAP: {OVERLAP} (fraction)")
    print(f"DUOMENU_APLANKAS: {str(DUOMENU_APLANKAS)}")
    print(f"PARAMETRU_APLANKAS: {str(PARAMETRU_APLANKAS)}")
    print(f"MODEL_PATH: {str(MODEL_PATH)}")
    print(f"CLEAN_ECG_DIR: {str(CLEAN_ECG_DIR)}")
    print(f"PORTION_LENGTH_IN_SECS: {PORTION_LENGTH_IN_SECS} seconds")
    print(f"MSE_THRESH: {MSE_THRESH}")
    print(f"FLAG_PLOT: {FLAG_PLOT}")
    print("=" * 40)

def main():
    
    # Print constants at the start
    print_constants()
    
    # Load model
    try:
        model = tf.keras.models.load_model(MODEL_PATH)
    except Exception as e:
        logger.error(f"Failed to load model from {MODEL_PATH}: {e}")
        return
    
    # Output directory for plotting results
    output_path = DUOMENU_APLANKAS / 'results'
    output_path.mkdir(exist_ok=True)
    
    results_for_all_files = {   
        'segments': [], 
        'mse_values': {level: [] for level, _ in NOISE_LEVELS},     
        'snr_values': {level: [] for level, _ in NOISE_LEVELS},
        'corr_values': {level: [] for level, _ in NOISE_LEVELS},
        'counts': {level: 0 for level, _ in NOISE_LEVELS}
    }
    
    logger.info("Starting ECG noise evaluation...")
    # Process each ECG file
    for ecg_file in CLEAN_ECG_DIR.glob('*.npy'):
        logger.info(f"Processing {ecg_file}")
        
        # Evaluate with random noise
        results, recon_signal = evaluate_noise(
            model, MSE_THRESH, ecg_file, 
            noise_option='random', 
            noise_probs=[0.4, 0.3, 0.2, 0.1]  # Example probabilities
        )
       
       # Print first 10 segments
        print("\nFirst 10 segments (if available):")
        for i, seg in enumerate(results['segments'][:10]):
            print(f"Segment {i+1}: Start={seg['start']}, End={seg['end']}, "
                  f"Noise Level={seg['noise_level']}, MSE={seg['mse']:.4f}, "
                  f"Is Noisy={seg['is_noisy']}")
        
        # Save results
        if FLAG_PLOT:
            visualize_results(recon_signal, results, output_path)
            print(f"\nResults visualized and saved to {output_path}")
        
        # Print statistics
        print(f"\nResults for {ecg_file.name}:")
        for level, _ in NOISE_LEVELS:
            count = results['counts'][level]
            mse_avg = np.mean(results['mse_values'][level]) if results['mse_values'][level] else 0
            snr_avg = np.mean(results['snr_values'][level]) if results['snr_values'][level] else 0
            corr_avg = np.mean(results['corr_values'][level]) if results['corr_values'][level] else 0
            print(f"{level}: Count={count}, MSE={mse_avg:.4f}, SNR={snr_avg:.2f}dB, CORR={corr_avg:.4f}")
        
        # Calculate confusion matrix and classification report
        if results['segments']:
            # True labels: 0 for clean, 1 for other (mild, moderate, severe)
            true_labels = [0 if seg['noise_level'] == 'clean' else 1 for seg in results['segments']]
            # Predicted labels: 0 for not noisy, 1 for noisy (based on MSE threshold)
            pred_labels = [1 if seg['is_noisy'] else 0 for seg in results['segments']]
            
            print("\nConfusion Matrix (rows: true [clean, other]; cols: pred [clean, other]):")
            cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1])
            print(cm)
            
            print("\nClassification Report:")
            print(classification_report(true_labels, pred_labels, target_names=['clean', 'other'], digits=4))
        else:
            print("\nNo segments available for confusion matrix and classification report.")

    # Collecting results for all files
    
    

if __name__ == "__main__":
    main()

2025-06-14 17:28:02.125054: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-14 17:28:02.514176: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-14 17:28:02.514456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-14 17:28:02.551351: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-14 17:28:02.663336: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-14 17:28:02.667739: I tensorflow/core/platform/cpu_feature_guard.cc:1

Script Constants:
NOISE_LEVELS:
  - clean: No parameters
  - mild:
      gaussian_std_ratio: 0.03
      baseline_amp: 0.05
      powerline_amp: 0.05
      motion_amp: 0.3
  - moderate:
      gaussian_std_ratio: 0.08
      baseline_amp: 0.12
      powerline_amp: 0.1
      motion_amp: 0.5
  - severe:
      gaussian_std_ratio: 0.15
      baseline_amp: 0.25
      powerline_amp: 0.15
      motion_amp: 1.2
BASELINE_FREQ: 0.33 Hz
POWERLINE_FREQ: 50 Hz
MOTION_PROB: 0.008
MOTION_DURATION_MIN: 10 samples
MOTION_DURATION_MAX: 100 samples
FS: 200 Hz (sampling frequency)
SEGMENT_LENGTH: 1024 samples
OVERLAP: 0.0 (fraction)
DUOMENU_APLANKAS: /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD
PARAMETRU_APLANKAS: /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD/PARAMETRAI
MODEL_PATH: /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD/PARAMETRAI/resunet_ecg_1024_05_31.keras
CLEAN_ECG_DIR: /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD/clean_ecg_for_test
PORTION_LENGTH_IN_SECS: 10 seconds
MSE_THRESH: 0.01
FLAG_PLOT: True


INFO:__main__:Processing /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD/clean_ecg_for_test/1102_1.npy



First 10 segments (if available):
Segment 1: Start=0, End=1024, Noise Level=clean, MSE=0.0024, Is Noisy=False
Segment 2: Start=1024, End=2048, Noise Level=clean, MSE=0.0016, Is Noisy=False
Segment 3: Start=4096, End=5120, Noise Level=moderate, MSE=0.0139, Is Noisy=True
Segment 4: Start=5120, End=6144, Noise Level=clean, MSE=0.0014, Is Noisy=False
Segment 5: Start=6144, End=7168, Noise Level=clean, MSE=0.0018, Is Noisy=False
Segment 6: Start=7168, End=8192, Noise Level=moderate, MSE=0.0130, Is Noisy=True
Segment 7: Start=8192, End=9216, Noise Level=moderate, MSE=0.0156, Is Noisy=True
Segment 8: Start=9216, End=10240, Noise Level=clean, MSE=0.0018, Is Noisy=False
Segment 9: Start=10240, End=11264, Noise Level=severe, MSE=0.0283, Is Noisy=True
Segment 10: Start=11264, End=12288, Noise Level=moderate, MSE=0.0134, Is Noisy=True

Results visualized and saved to /home/kesju/DI/ZIVEO_2025/DUOMENYS_UPD/results

Results for 1102_1.npy:
clean: Count=55, MSE=0.0021, SNR=infdB, CORR=1.0000
mild: C